In [1]:
# Cell 1 — Setup (same preprocessing as 02_baseline, condensed)
import json
import pathlib
import datetime
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier

# Load + fix mixed-type columns
df = pd.read_csv('../data/raw/hotel_bookings.csv')
df['agent'] = df['agent'].fillna('Unknown').astype(str)
df['company'] = df['company'].fillna('Unknown').astype(str)
df = df.drop(columns=['reservation_status', 'reservation_status_date'])

X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessor
numeric_cols = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'adr',
    'required_car_parking_spaces', 'total_of_special_requests'
]
categorical_cols = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel',
    'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'agent', 'company', 'customer_type'
]

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_cols)
])

# Simple run logger — saves every experiment to results/runs.json
RUNS_FILE = pathlib.Path('../results/runs.json')
RUNS_FILE.parent.mkdir(exist_ok=True)

def log_run(name, params, f1):
    runs = json.loads(RUNS_FILE.read_text()) if RUNS_FILE.exists() else []
    runs.append({'name': name, 'params': params, 'f1': round(f1, 4),
                 'timestamp': datetime.datetime.now().isoformat()})
    RUNS_FILE.write_text(json.dumps(runs, indent=2))
    print(f'Logged run "{name}" → F1={f1:.4f}')

# Seed with the Day 20 LR baseline so the comparison is complete
if not RUNS_FILE.exists():
    log_run('logistic_regression_baseline', {}, 0.7590)

print('Setup complete — train:', X_train.shape[0], '| test:', X_test.shape[0])

Logged run "logistic_regression_baseline" → F1=0.7590
Setup complete — train: 95512 | test: 23878


In [2]:
# Cell 2 — Random Forest
#
# Random Forest = 100 decision trees, each trained on a random subset of rows + features.
# Final prediction = majority vote across all 100 trees.
# Captures non-linear relationships LR misses (e.g. lead_time > 200 AND deposit = Non Refund).
# Expect 1-3 minutes.

params_rf = dict(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(**params_rf))
])

print('Fitting Random Forest... (1-3 minutes)')
rf_pipeline.fit(X_train, y_train)
print('Done!')

y_pred_rf = rf_pipeline.predict(X_test)
f1_rf = f1_score(y_test, y_pred_rf)
log_run('random_forest_baseline', params_rf, f1_rf)

print(f'\nRandom Forest F1: {f1_rf:.4f}  (LR baseline was 0.7590)')
print()
print(classification_report(y_test, y_pred_rf, target_names=['Not cancelled', 'Cancelled']))

Fitting Random Forest... (1-3 minutes)
Done!
Logged run "random_forest_baseline" → F1=0.8476

Random Forest F1: 0.8476  (LR baseline was 0.7590)

               precision    recall  f1-score   support

Not cancelled       0.89      0.94      0.92     15033
    Cancelled       0.89      0.81      0.85      8845

     accuracy                           0.89     23878
    macro avg       0.89      0.87      0.88     23878
 weighted avg       0.89      0.89      0.89     23878



In [3]:
# Cell 3 — XGBoost
#
# XGBoost = gradient boosted trees. Builds trees SEQUENTIALLY — each one corrects
# the previous one's mistakes. Generally outperforms RF on tabular data.
# Expect 1-2 minutes.

params_xgb = dict(
    n_estimators=100, learning_rate=0.1,
    max_depth=6, random_state=42, n_jobs=-1
)

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(**params_xgb))
])

print('Fitting XGBoost... (1-2 minutes)')
xgb_pipeline.fit(X_train, y_train)
print('Done!')

y_pred_xgb = xgb_pipeline.predict(X_test)
f1_xgb = f1_score(y_test, y_pred_xgb)
log_run('xgboost_baseline', params_xgb, f1_xgb)

print(f'\nXGBoost F1: {f1_xgb:.4f}  (RF was {f1_rf:.4f}, LR was 0.7590)')
print()
print(classification_report(y_test, y_pred_xgb, target_names=['Not cancelled', 'Cancelled']))

Fitting XGBoost... (1-2 minutes)
Done!
Logged run "xgboost_baseline" → F1=0.8157

XGBoost F1: 0.8157  (RF was 0.8476, LR was 0.7590)

               precision    recall  f1-score   support

Not cancelled       0.88      0.92      0.90     15033
    Cancelled       0.86      0.78      0.82      8845

     accuracy                           0.87     23878
    macro avg       0.87      0.85      0.86     23878
 weighted avg       0.87      0.87      0.87     23878



In [ ]:
# Cell 4 — GridSearchCV on XGBoost
#
# Tries every combination of hyperparameters, picks the best via 3-fold cross-validation
# on the TRAINING set. Test set stays untouched.
# 2×2×2 = 8 combos × 3 folds = 24 fits. Expect 5-10 minutes.

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth':    [4, 6],
    'classifier__learning_rate':[0.05, 0.1]
}

xgb_base = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, n_jobs=-1))
])

grid_search = GridSearchCV(
    xgb_base, param_grid,
    scoring='f1', cv=3, n_jobs=-1, verbose=1
)

print('Running GridSearchCV (24 fits)... may take 5-10 minutes')
grid_search.fit(X_train, y_train)
print('Done!')

best_params  = grid_search.best_params_
best_pipeline = grid_search.best_estimator_
y_pred_best  = best_pipeline.predict(X_test)
f1_best      = f1_score(y_test, y_pred_best)
log_run('xgboost_tuned', best_params, f1_best)

print(f'\nBest params: {best_params}')
print(f'Tuned XGBoost F1: {f1_best:.4f}  (untuned XGB was {f1_xgb:.4f})')
print()
print(classification_report(y_test, y_pred_best, target_names=['Not cancelled', 'Cancelled']))

Running GridSearchCV (24 fits)... may take 5-10 minutes
Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [ ]:
# Cell 5 — Model comparison summary
runs = json.loads(RUNS_FILE.read_text())
runs_sorted = sorted(runs, key=lambda r: r['f1'])

print('=== Model Comparison (F1 score on test set) ===')
for r in runs_sorted:
    bar = '█' * int(r['f1'] * 40)
    print(f"{r['name']:<35} {r['f1']:.4f}  {bar}")

winner = max(runs, key=lambda r: r['f1'])
print(f"\nWinner: {winner['name']} → F1={winner['f1']:.4f}")
print('This model goes forward to SHAP + Gradio on Days 22-23.')